# Один фиксированный LightGBM на Kaggle

По умолчанию выбран только 015_tree, STAGE="fixed", SMOKE=False: один fit на всём train, 855 признаков, 420 итераций, затем val. Screening, search, CV и early stopping не запускаются. Подключите готовый cache/features и исходные метки через Input; folds для этого прогона не нужны. Код новой версии должен быть опубликован и доступен по CODE_REF. Экспорт — одним ZIP в последней ячейке.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/GreatDruk/sleep-edf-classification.git"
CODE_REF = "feat/boosting"  # Лучше SHA коммита для полноценного эксперимента.
PROJECT_ROOT = Path("/tmp/sleep-edf-code")
OUTPUT_ROOT = Path("/kaggle/working/artefacts")
SMOKE = False
FAMILY = "tree"
STAGE = "fixed"
THREADS = "auto"
CACHE_ROOT = None  # Корень готового artefacts/cache/features в Input.
FOLD_ROOT = None  # Корень сохранённых splits в Input.
CONFIG_IDS = (
    ["015_tree"]
    if FAMILY == "tree"
    else [f"{i:03d}_linear" for i in (1, 2, 3, 4, 5, 7)]
)


def checkout_code():
    if PROJECT_ROOT.exists():
        current_ref = subprocess.check_output(
            ["git", "config", "sleepedf.bootstrapRef"], cwd=PROJECT_ROOT, text=True
        ).strip()
        if current_ref != CODE_REF:
            raise RuntimeError(
                "Для другого CODE_REF задайте новый PROJECT_ROOT в /tmp."
            )
        print(
            "Используется существующий checkout; для обновления ветки задайте новый PROJECT_ROOT."
        )
        return
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="sleep-git-") as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] else os.environ["SLEEP_GITHUB_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "SLEEP_GITHUB_TOKEN": token,
        }
        subprocess.run(
            ["git", "init", "--quiet", "--initial-branch=bootstrap", str(PROJECT_ROOT)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "add", "origin", REPO_URL], cwd=PROJECT_ROOT, check=True
        )
        subprocess.run(
            [
                "git",
                "-c",
                "credential.helper=",
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                CODE_REF,
            ],
            cwd=PROJECT_ROOT,
            env=env,
            check=True,
        )
        subprocess.run(
            ["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        subprocess.run(
            ["git", "config", "sleepedf.bootstrapRef", CODE_REF],
            cwd=PROJECT_ROOT,
            check=True,
        )
    del token


checkout_code()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print("Код:", commit)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook import run_command  # noqa: E402

In [ ]:
# Устанавливаем только инструмент uv; зависимости проекта устанавливает uv.lock.
UV_VERSION = "0.12.15"
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"], check=True
    )
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv не найден в PATH после установки")
ENV = {
    "UV_PROJECT_ENVIRONMENT": "/tmp/sleep-edf-venv",
    "UV_CACHE_DIR": "/tmp/sleep-uv-cache",
    "UV_PYTHON_INSTALL_DIR": "/tmp/sleep-uv-python",
}
print("Ядро Kaggle:", sys.version)
run_command([UV, "--version"], cwd=PROJECT_ROOT, env=ENV)
run_command(
    [UV, "sync", "--locked", "--no-dev", "--python", "3.14.2"],
    cwd=PROJECT_ROOT,
    env=ENV,
)
run_command(
    [
        UV,
        "run",
        "--locked",
        "--no-dev",
        "python",
        "-c",
        "import sys, sklearn, numpy; print(sys.version); print('sklearn', sklearn.__version__, 'numpy', numpy.__version__)",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)

In [ ]:
# Ищем корень подключённой поставки по меткам и каталогам записей.
roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("labels.csv")
    if all((p.parent / "epochs" / split).is_dir() for split in ("train", "val", "test"))
]
if len(roots) != 1:
    raise RuntimeError(
        f"Нужен ровно один dataset full_psg_v1; найдены: {roots}. Проверьте Add Input."
    )
DATA_ROOT = roots[0]
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT, "SMOKE:", SMOKE)
print("Записей train:", len(list((DATA_ROOT / "epochs/train").glob("*.npz"))))
if FAMILY == "tree":
    if CACHE_ROOT is None:
        caches = [
            p
            for p in Path("/kaggle/input").rglob("features")
            if p.is_dir() and list(p.glob("*/definition.json"))
        ]
        if len(caches) != 1:
            raise FileNotFoundError(
                "Подключите готовый кэш признаков через Input и укажите CACHE_ROOT."
            )
        CACHE_ROOT = caches[0]
    CACHE_ROOT = Path(CACHE_ROOT)
    FOLD_ROOT = (
        Path(FOLD_ROOT) if FOLD_ROOT is not None else CACHE_ROOT.parents[1] / "splits"
    )
    print("CACHE_ROOT:", CACHE_ROOT, "FOLD_ROOT:", FOLD_ROOT)

## Единственный запуск

Убедитесь, что CONFIG_IDS=["015_tree"], STAGE="fixed", SMOKE=False. Эта ячейка выполняет одно обучение LightGBM на всём train и оценивает val после fit. Число итераций фиксировано — 420; данные val не передаются в fit. Сохраняется только fixed-модель, OOF не создаётся. Кэш читается без пересчёта признаков.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Сохраняем лёгкую копию исходников для последующего инференса: без .git и окружения.
source_snapshot = OUTPUT_ROOT / "source"
if source_snapshot.exists():
    saved_commit = (source_snapshot / "commit.txt").read_text().strip()
    if saved_commit != commit:
        raise RuntimeError(
            "В этом OUTPUT_ROOT другая версия кода; выберите новый каталог результатов."
        )
else:
    source_snapshot.mkdir()
    for name in ("src", "configs"):
        shutil.copytree(
            PROJECT_ROOT / name,
            source_snapshot / name,
            ignore=shutil.ignore_patterns("__pycache__"),
        )
    for name in ("pyproject.toml", "uv.lock", ".python-version", "experiments.csv"):
        shutil.copy2(PROJECT_ROOT / name, source_snapshot / name)
    (source_snapshot / "commit.txt").write_text(commit + "\n")
command = [
    UV,
    "run",
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.run",
    *[str(PROJECT_ROOT / "configs" / f"{name}.yaml") for name in CONFIG_IDS],
    "--data-root",
    str(DATA_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
]
if FAMILY == "tree":
    command += [
        "--stage",
        STAGE,
        "--threads",
        str(THREADS),
        "--cache-root",
        str(CACHE_ROOT),
        "--fold-root",
        str(FOLD_ROOT),
    ]
if SMOKE:
    command.append("--smoke")
try:
    run_command(command, cwd=PROJECT_ROOT, env=ENV)
finally:
    shutil.copy2(PROJECT_ROOT / "experiments.csv", OUTPUT_ROOT / "experiments.csv")

In [ ]:
namespace = OUTPUT_ROOT / "smoke" if SMOKE else OUTPUT_ROOT
for name in CONFIG_IDS:
    path = namespace / name / "metrics.json"
    if path.exists():
        metrics = json.loads(path.read_text())
        for regime, values in metrics["val"].items():
            print(name, regime, "val macro-F1:", round(values["macro_f1"], 5))
    else:
        print(name, "ещё не завершён")

## Скачать результаты

Следующая ячейка создаёт один архив со всем OUTPUT_ROOT: модели, метрики, OOF/val-вероятности, trials, folds, журнал, кэши и снимок кода. Окружение и исходные сигналы в архив не входят. После завершения нажмите ссылку — скачивать отдельные файлы не требуется.

Для фоновой серии используйте Save Version → Save & Run All. ZIP окажется в Output notebook. Приватный код входит в архив, поэтому notebook и outputs сохраняйте приватными. Между сессиями сохраняйте outputs; наличие файлов в интерактивной сессии само по себе не гарантирует их сохранение.


In [ ]:
from datetime import UTC, datetime

from IPython.display import FileLink, display

archive = Path("/kaggle/working") / (
    "sleep-edf-results-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ") + ".zip"
)
export_command = [
    UV,
    "run",
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.export",
    str(OUTPUT_ROOT),
    "--archive",
    str(archive),
]
if FAMILY == "tree":
    export_command += ["--cache-root", str(CACHE_ROOT)]
    if FOLD_ROOT is not None and Path(FOLD_ROOT).is_dir():
        export_command += ["--fold-root", str(FOLD_ROOT)]
run_command(export_command, cwd=PROJECT_ROOT, env=ENV)
# FileLink должен быть относительно корня, обслуживаемого Kaggle.
display(
    FileLink(
        str(archive.relative_to(Path.cwd())),
        result_html_prefix="Скачать все результаты: ",
    )
)